In [1]:
import os
import time
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

In [2]:
import sys
from pathlib import Path

current_dir = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    raise RuntimeError('Could not locate repo root (pyproject.toml + bcosgnn/).')

project_root = find_repo_root(current_dir)

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Repo root added: {project_root}")

Repo root added: /home/moso00002/Desktop/gnn/bcosgnn-bcos_gnn_shaique


In [3]:
class GINE4Layer(nn.Module):
    def __init__(
        self,
        in_dim: int,
        edge_dim: int,
        hidden_dim: int,
        num_classes: int,
        num_layers: int = 4,
        dropout: float = 0.2,
    ):
        super().__init__()
        assert num_layers == 4, "This notebook cell implements the requested 4-layer GINE."
        self.dropout = dropout

        self.node_encoder = nn.Linear(in_dim, hidden_dim)
        self.edge_encoder = nn.Linear(edge_dim, hidden_dim)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(inplace=True),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(nn=mlp))
            self.norms.append(nn.BatchNorm1d(hidden_dim))

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.node_encoder(x)
        edge_attr = self.edge_encoder(edge_attr)

        for conv, bn in zip(self.convs, self.norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x, inplace=True)
            x = F.dropout(x, p=self.dropout, training=self.training)

        g = global_mean_pool(x, batch)
        return self.classifier(g)

In [4]:
import torch
import os
from pathlib import Path
from torch_geometric.data import InMemoryDataset
from torch_geometric.loader import DataLoader

class ZincProcessedDataset(InMemoryDataset):
    def __init__(self, root, transform=None, pre_transform=None):
        super().__init__(root, transform, pre_transform)
        
        # Explicitly constructing the path to ensure it is correct
        data_path = Path(self.processed_dir) / "data.pt"
        
        print(f"Loading data from: {data_path.resolve()}")
        
        try:
             # weights_only=False required for PyTorch 2.6+ for PyG Data objects
            self.data, self.slices = torch.load(data_path, weights_only=False)
        except TypeError: 
            # Fallback for older PyTorch versions
            self.data, self.slices = torch.load(data_path)

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return ['data.pt']

    def download(self):
        pass

    def process(self):
        pass

cwd = Path.cwd()
print(f"Current Working Directory: {cwd}")


if (cwd / "zinc_di_halo_benzene_data").exists():
    dataset_path = "zinc_di_halo_benzene_data"
elif (cwd / "shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data").exists():
    dataset_path = "shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data"
else:

    potential_path = cwd / "shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data"
    dataset_path = str(potential_path)

print(f"Using dataset path: {dataset_path}")

try:
    dataset = ZincProcessedDataset(root=dataset_path)
    print(f"Dataset loaded: {len(dataset)} graphs")
    print(f"Number of features: {dataset.num_features}")
    print(f"Number of classes: {dataset.num_classes}")

    # Create DataLoader
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    print(f"DataLoader created with {len(loader)} batches")

except Exception as e:
    print(f"Error loading dataset: {e}")


Current Working Directory: /home/moso00002/Desktop/gnn/bcosgnn-bcos_gnn_shaique/shaique_updates/codes/multi_class_Zinc
Using dataset path: zinc_di_halo_benzene_data
Loading data from: /home/moso00002/Desktop/gnn/bcosgnn-bcos_gnn_shaique/shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data/processed/data.pt
Dataset loaded: 9000 graphs
Number of features: 10
Number of classes: 9
DataLoader created with 282 batches


In [5]:
from sklearn.model_selection import train_test_split
import numpy as np

labels = dataset.y.numpy()

print(np.unique(labels), np.bincount(labels))

train_size = 0.8
val_size = 0.1
test_size = 0.1

train_idx , tmp_idx = train_test_split(np.arange(len(labels)), stratify=labels, train_size=train_size, random_state=42)

val_idx , test_idx = train_test_split(tmp_idx, test_size= 0.5, stratify=labels[tmp_idx], random_state=42)

train_dataset = dataset[train_idx]
val_dataset = dataset[val_idx]
test_dataset = dataset[test_idx]

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

[0 1 2 3 4 5 6 7 8] [1000 1000 1000 1000 1000 1000 1000 1000 1000]
Train dataset size: 7200
Validation dataset size: 900
Test dataset size: 900


In [6]:
from collections import Counter

def check_balance(subset, name):
    labels = [data.y.item() for data in subset]
    counts = Counter(labels)
    print(f"--- {name} Class Distribution ---")
    # Print first few classes to save space
    print(sorted(counts.items())[:5]) 

check_balance(train_dataset, "Train")
check_balance(val_dataset, "Val")
check_balance(test_dataset, "Test")

--- Train Class Distribution ---
[(0, 800), (1, 800), (2, 800), (3, 800), (4, 800)]
--- Val Class Distribution ---
[(0, 100), (1, 100), (2, 100), (3, 100), (4, 100)]
--- Test Class Distribution ---
[(0, 100), (1, 100), (2, 100), (3, 100), (4, 100)]


In [7]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. Determine Input Dimensions
sample_data = dataset[0]
in_dim = dataset.num_features
# GINE requires edge attributes. Check if they exist and their dimension.
if sample_data.edge_attr is not None:
    edge_dim = sample_data.edge_attr.shape[1]
else:
    # If no edge attributes, GINE might fail or we might need to use 0 if supported/modified
    # Assuming appropriate edge_dim based on previous context or handling
    edge_dim = 0 
    print("Warning: No edge attributes found. GINEConv requires edge features.")

print(f"Input Feature Dim: {in_dim}")
print(f"Edge Feature Dim: {edge_dim}")
print(f"Number of Classes: {dataset.num_classes}")

# 2. Initialize Model
model = GINE4Layer(
    in_dim=in_dim,
    edge_dim=edge_dim,
    hidden_dim=64,
    num_classes=dataset.num_classes,
    num_layers=4,
    dropout=0.2
).to(device)

print(model)

# 3. Setup Training
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

def train():
    model.train()
    total_loss = 0
    total_samples = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
        total_samples += data.num_graphs
    return total_loss / total_samples

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0
    correct = 0
    total_samples = 0
    for data in loader:
        data = data.to(device)
        out = model(data)
        pred = out.argmax(dim=1)
        correct += int((pred == data.y).sum())
        loss = criterion(out, data.y)
        total_loss += loss.item() * data.num_graphs
        total_samples += data.num_graphs
    return total_loss / total_samples, correct / total_samples

# 4. Training Loop
epochs = 50
best_val_acc = 0.0
print("-" * 50)
print(f"Starting training for {epochs} epochs...")

for epoch in range(1, epochs + 1):
    train_loss = train()
    val_loss, val_acc = evaluate(val_loader)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Save best model
        torch.save(model.state_dict(), 'best_gine_model.pt')
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

print(f"Training finished. Best Val Acc: {best_val_acc:.4f}")

# 5. Inference on Test Set
print("-" * 50)
print("Loading best model for inference on Test Set...")
model.load_state_dict(torch.load('best_gine_model.pt'))
test_loss, test_acc = evaluate(test_loader)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Using device: cuda
Input Feature Dim: 10
Edge Feature Dim: 4
Number of Classes: 9
GINE4Layer(
  (node_encoder): Linear(in_features=10, out_features=64, bias=True)
  (edge_encoder): Linear(in_features=4, out_features=64, bias=True)
  (convs): ModuleList(
    (0-3): 4 x GINEConv(nn=Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
  )
  (norms): ModuleList(
    (0-3): 4 x BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU(inplace=True)
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=9, bias=True)
  )
)
--------------------------------------------------
Starting training for 50 epochs...
Epoch: 001, Train Loss: 1.3432, Val Loss: 0.8168, Val Acc: 0.8000
Epoch: 001, Train Loss: 1.3432, Val Loss: 0.8168

In [9]:
# ---------------------------------------------------------
# Check for Ground Truth Motifs / Explanations
# ---------------------------------------------------------

def inspect_ground_truth(dataset, num_samples=5):
    print("Inspecting for ground truth motif attributes...")
    
    # Get a few random samples
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    found_gt = False
    gt_attributes = []
    
    sample = dataset[0]
    # Fixed: Calling keys() properly
    print(f"Keys available in Data object: {sample.keys()}")
    
    # Common names for ground truth masks in GNN explanation datasets
    potential_gt_keys = ['edge_mask', 'node_mask', 'explanation', 'z', 'gt_mask']
    
    for key in potential_gt_keys:
        if hasattr(sample, key) and getattr(sample, key) is not None:
            found_gt = True
            gt_attributes.append(key)
    
    # Removed the redundant and erroneous check that was causing TypeError

    if found_gt:
        print(f"Potential Ground Truth attributes found: {list(set(gt_attributes))}")
        
        # Visualize statistics of the mask for a few samples
        for idx in indices:
            data = dataset[idx]
            print(f"\n--- Graph {idx} ---")
            print(f"Num Nodes: {data.num_nodes}, Num Edges: {data.num_edges}")
            
            if hasattr(data, 'edge_mask'):
                mask = data.edge_mask
                print(f"Edge Mask shape: {mask.shape}")
                print(f"Non-zero elements in edge_mask: {mask.sum().item()}")
                print(f"Sparsity: {mask.sum().item() / mask.numel():.4f}")
                
            if hasattr(data, 'node_mask'):
                 mask = data.node_mask
                 print(f"Node Mask shape: {mask.shape}")
                 print(f"Non-zero elements in node_mask: {mask.sum().item()}")

    else:
        print("No standard ground truth motif attributes ('edge_mask', 'node_mask') found directly on the Data objects.")
        print("Attempting to infer from other attributes if available...")


inspect_ground_truth(dataset)

Inspecting for ground truth motif attributes...
Keys available in Data object: ['edge_index', 'explanation_mask', 'x', 'y', 'edge_attr']
No standard ground truth motif attributes ('edge_mask', 'node_mask') found directly on the Data objects.
Attempting to infer from other attributes if available...


In [10]:
def verify_explanation_mask(dataset):
    print("Verifying 'explanation_mask'...")
    data = dataset[0]
    mask = data.explanation_mask
    
    print(f"Num Edges: {data.num_edges}")
    print(f"Num Nodes: {data.num_nodes}")
    print(f"Mask Shape: {mask.shape}")
    
    if mask.shape[0] == data.num_edges:
        print(">> Confirmed: 'explanation_mask' corresponds to EDGES.")
    elif mask.shape[0] == data.num_nodes:
        print(">> Confirmed: 'explanation_mask' corresponds to NODES.")
    else:
        print(">> Warning: Mask shape does not match nodes or edges directly.")

    # Cast to float for statistics, as mask is likely Long/Int
    mask_float = mask.float()
    print(f"Mask statistics: Min {mask_float.min():.4f}, Max {mask_float.max():.4f}, Mean {mask_float.mean():.4f}")
    
    # Check unique values to see if it's binary or categorical
    print(f"Unique values in mask: {torch.unique(mask)}")
    
verify_explanation_mask(dataset)

Verifying 'explanation_mask'...
Num Edges: 80
Num Nodes: 39
Mask Shape: torch.Size([39])
>> Confirmed: 'explanation_mask' corresponds to NODES.
Mask statistics: Min 0.0000, Max 1.0000, Mean 0.2308
Unique values in mask: tensor([0, 1])
